# v0.12.0 -- `retry_on_conflict` & optimistic concurrency

Under SurrealDB's optimistic concurrency, a transaction is rolled back with a **retryable** conflict when a concurrent writer changes the same data. v0.12.0 adds:

- **`retry_on_conflict(...)`** -- an async decorator that re-runs the function (a fresh transaction per attempt) with exponential backoff + jitter, but **only** on a real conflict.
- **`SurrealDbConflictError`** -- one typed exception for a retryable conflict, raised on **both** transaction strategies.
- **`is_conflict_error(exc)`** -- the public predicate the decorator uses.

**Same on SurrealDB 2.6.x and 3.x**: the exception type and the decorator are identical on both lines. Only the _frequency_ of conflicts differs -- on 3.x (optimistic MVCC) conflicts are the normal failure mode; on 2.6.x the engine serialises more, so they are rarer.

## 1. Connect

WebSocket (`.../rpc`) so native interactive transactions are used on SurrealDB 3.x.

In [1]:
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Which transaction strategy?

Probe the server's native transaction RPC -- present on SurrealDB 3.x (interactive), absent on 2.6.x (buffered). The notebook runs end-to-end either way.

In [2]:
import contextlib


async def native_tx_supported() -> bool:
    client = await SurrealDBConnectionManager.get_client()
    try:
        txn = await client.begin()
    except Exception:
        return False
    with contextlib.suppress(Exception):
        await client.cancel(txn)
    return True


interactive = await native_tx_supported()
print("Interactive transactions available (SurrealDB 3.x):", interactive)

Interactive transactions available (SurrealDB 3.x): True


## 3. A model + a clean slate

In [3]:
from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Account(BaseSurrealModel):
    model_config = SurrealConfigDict(primary_key="id")
    id: str | None = None
    balance: int


client = await SurrealDBConnectionManager.get_client()
for table in ("Account", "counter"):
    with contextlib.suppress(Exception):
        await client.query(f"DELETE {table};", {})
print("clean slate ready")

clean slate ready


## 4. Detecting a conflict

`is_conflict_error` anchors on SurrealDB's own retryable marker -- the phrase *"This transaction can be retried"* (present on both 2.6.5 and 3.1.3). A non-retryable failure (e.g. a duplicate-key error) is **not** treated as a conflict, so it is never retried pointlessly.

In [4]:
from surreal_orm_lite import SurrealDbConflictError, SurrealDbError, is_conflict_error

retryable = SurrealDbError(
    "Transaction conflict: Write conflict, retry the transaction. This transaction can be retried"
)
duplicate = SurrealDbError("Database record `Account:alice` already exists")

print("retryable conflict? ", is_conflict_error(retryable))
print("duplicate-key?      ", is_conflict_error(duplicate))
print("typed conflict?     ", is_conflict_error(SurrealDbConflictError("x")))

retryable conflict?  True
duplicate-key?       False
typed conflict?      True


## 5. `retry_on_conflict` -- the happy path

Wrap a function that opens its own `transaction()`. With no conflict it runs once and commits. Total attempts = `max_retries + 1`.

In [5]:
from surreal_orm_lite import retry_on_conflict


@retry_on_conflict(max_retries=3, base_delay=0.01, jitter=False)
async def open_account(acc_id: str, balance: int) -> str:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Account(id=acc_id, balance=balance).save(tx=tx)
    return acc_id

await open_account("alice", 100)
await open_account("bob", 50)
rows = await Account.objects().all()
print("accounts:", sorted((r.get_raw_id(), r.balance) for r in rows))

accounts: [('alice', 100), ('bob', 50)]


## 6. Retry in action

Simulate a concurrent writer that wins the race on the **first** attempt: the function raises a `SurrealDbConflictError`, the transaction rolls back, and `retry_on_conflict` re-runs it. The second attempt succeeds and commits. Runs the same on both DB lines (no reads inside the transaction, so the buffered strategy is happy too).

In [6]:
attempts = {"n": 0}


@retry_on_conflict(max_retries=5, base_delay=0.01, jitter=False)
async def claim(acc_id: str) -> int:
    attempts["n"] += 1
    async with SurrealDBConnectionManager.transaction() as tx:
        await Account(id=acc_id, balance=attempts["n"]).save(tx=tx)
        if attempts["n"] == 1:
            raise SurrealDbConflictError(
                "Transaction conflict: Write conflict, retry the transaction. "
                "This transaction can be retried"
            )
    return attempts["n"]

winning_attempt = await claim("carol")
print(f"committed on attempt #{winning_attempt}")
carol = await Account.objects().get("carol")
print("carol.balance:", carol.balance)

Transaction conflict in claim (attempt 1/5), retrying in 0.010s...


committed on attempt #2
carol.balance: 2


## 7. Giving up after the retries are exhausted

If every attempt conflicts, the last conflict is re-raised as `SurrealDbConflictError` (its message says how many retries were spent).

In [7]:
@retry_on_conflict(max_retries=2, base_delay=0.01, jitter=False)
async def always_conflicts() -> None:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Account(id="ghost", balance=0).save(tx=tx)
        raise SurrealDbConflictError(
            "Transaction conflict: Write conflict, retry the transaction. "
            "This transaction can be retried"
        )

try:
    await always_conflicts()
except SurrealDbConflictError as exc:
    print("gave up:", exc)

Transaction conflict in always_conflicts (attempt 1/2), retrying in 0.010s...


Transaction conflict in always_conflicts (attempt 2/2), retrying in 0.020s...


gave up: Transaction conflict persisted after 2 retries in always_conflicts: Transaction conflict: Write conflict, retry the transaction. This transaction can be retried


## 8. A real server conflict (SurrealDB 3.x)

On the interactive line we can force a genuine write/write conflict with two concurrent transactions on separate connections -- the loser's `commit()` raises a real `SurrealDbConflictError` straight from the server. On the buffered line (2.6.x / HTTP) the cell explains the difference instead of erroring.

In [8]:
from surreal_orm_lite._sdk import AsyncSurreal
from surreal_orm_lite.transaction import InteractiveTransaction


async def _raw():
    url = f"ws://{HOST}:{PORT}/rpc"
    db = AsyncSurreal(url)
    await db.connect(url)
    await db.signin({"username": "root", "password": "root"})
    await db.use("examples", "examples")
    return db


if interactive:
    with contextlib.suppress(Exception):
        await client.query("DELETE counter:c;", {})
    await client.query("CREATE counter:c SET n = 0;", {})
    c1 = await _raw()
    c2 = await _raw()
    try:
        t1 = await c1.begin()
        t2 = await c2.begin()
        tx1 = InteractiveTransaction(c1, t1)
        tx2 = InteractiveTransaction(c2, t2)
        await tx1.run_read("SELECT * FROM counter:c;", {})
        await tx2.run_read("SELECT * FROM counter:c;", {})
        await tx1.add("UPDATE counter:c SET n = 1;", {})
        await tx2.add("UPDATE counter:c SET n = 2;", {})
        await tx1.commit()  # wins
        try:
            await tx2.commit()  # loses
            print("no conflict (unexpected)")
        except SurrealDbConflictError as exc:
            print("server raised SurrealDbConflictError:")
            print(" ", exc)
    finally:
        for c in (c1, c2):
            with contextlib.suppress(Exception):
                await c.close()
else:
    print(
        "Buffered line (SurrealDB 2.6.x / HTTP): conflicts are rarer (the engine "
        "serialises more) but still surface as SurrealDbConflictError when they occur."
    )

server raised SurrealDbConflictError:
  Transaction conflict: Write conflict, retry the transaction. This transaction can be retried


## 9. Cleanup

In [9]:
for table in ("Account", "counter"):
    with contextlib.suppress(Exception):
        await client.query(f"DELETE {table};", {})
await SurrealDBConnectionManager.close_connection()
print("done")

done
